In [1]:
import spacy
import re
import requests

nlp = spacy.load("en_core_web_sm")

In [28]:
text = """Continuing your research into claim extraction, here is a factual summary concerning Shah Rukh Khan as of today, April 16, 2026. This paragraph is designed to offer a mix of professional, personal, and sporting data points to challenge an extraction model.Celebrity & Business: Shah Rukh Khan’s 2026 LandscapeAs of Thursday, April 16, 2026, superstar Shah Rukh Khan is navigating a high-stakes period in both cinema and professional sports. In the Indian Premier League (IPL) 2026, his franchise, the Kolkata Knight Riders (KKR), is facing a critical juncture after losing to the Chennai Super Kings on April 14, leaving them at the bottom of the points table with only one point from five matches. On the cinematic front, Khan has officially slated his next major film, King, for a theatrical release on December 24, 2026. Directed by Siddharth Anand, the action-thriller features an ensemble cast including Abhishek Bachchan, Deepika Padukone, and notably marks the big-screen debut of his daughter, Suhana Khan. To maintain his silver-haired "look" for the role, Khan reportedly declined a cameo in the upcoming sequel Jailer 2. Additionally, interest in his personal brand spiked today following an interview with his former bodyguard, Yaseen Khan, who shared that the actor previously maintained eight different passports due to the frequency of his international filming schedules.Claim Extraction BenchmarksFor your research, you can evaluate your model's ability to extract these specific types of claims from the text:Quantitative Performance Claim: KKR has earned one point in five matches during the 2026 season.Predictive/Scheduling Claim: The movie King will release on December 24, 2026.Relational Claim: Suhana Khan is debuting alongside her father in a project directed by Siddharth Anand.Decision-Based Claim: Shah Rukh Khan rejected a role in Jailer 2 to protect his aesthetic continuity for King.Biographical Claim: Former aide Yaseen Khan claims the actor used eight passports for global travel.Does this variety of sports, entertainment, and personal claims help in refining your extraction parameters?"""

In [29]:
def build_prompt(text):
    return f"""
You are a high-precision factual claim extraction system.

Extract ONLY factual claims explicitly stated in the text.

STRICT RULES:

1. Only extract facts directly present in the text
2. DO NOT infer or add new information
3. DO NOT use placeholders like "not specified", "unknown"
4. Each claim must contain ONLY ONE fact
5. Replace pronouns with real entity names
6. Do NOT use vague subjects like "the company", "the bill"
7. Each claim must follow: SUBJECT + RELATION + OBJECT
8. Skip unclear or ambiguous statements
9. Do NOT include opinions or general statements
10.Always include full entity type when needed (e.g., "the film King",, "an interview featuring Yaseen Khan","actor sharukh")
11.Avoid generic phrasing like "is misleading"; instead use "stated that" or "reported that"

OUTPUT:
- Only numbered claims
- No explanations
- No extra text

TEXT:
{text}
"""

In [30]:
def llm_extract_claims(text):

    prompt = build_prompt(text)

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3",
                "prompt": prompt,
                "stream": False
            },
            timeout=60
        )

        if response.status_code != 200:
            print("❌ API Error:", response.text)
            return []

        data = response.json()

        output = data.get("response") or data.get("message", {}).get("content", "")

        if not output:
            print("⚠️ Empty response:", data)
            return []

        claims = []
        for line in output.split("\n"):
            line = line.strip()

            if not line:
                continue

            # remove numbering
            line = re.sub(r'^\d+[\).\-\s]*', '', line)

            claims.append(line)

        return claims

    except Exception as e:
        print("❌ Error:", str(e))
        return []

In [31]:
def clean_claims(claims):

    cleaned = []

    for c in claims:
        c_lower = c.lower()

        # remove LLM meta text
        if any(x in c_lower for x in [
            "here is", "here are", "note:", "output", "format"
        ]):
            continue

        # remove short / weak claims
        if len(c.split()) < 5:
            continue

        cleaned.append(c.strip())

    return list(set(cleaned))

In [32]:
def remove_hallucinations(claims):

    valid = []

    for c in claims:
        c_lower = c.lower()

        if any(x in c_lower for x in [
            "not specified", "unknown", "unclear"
        ]):
            continue

        valid.append(c)

    return valid

In [33]:
def strict_validation(claims, text):

    validated = []
    text_lower = text.lower()

    for c in claims:

        words = c.lower().split()

        overlap = sum(1 for w in words if w in text_lower)

        if overlap < 4:
            continue

        if c.lower().startswith(("this", "these", "those")):
            continue

        validated.append(c)

    return list(set(validated))

In [34]:
def score_claim(claim):

    doc = nlp(claim)

    entity_score = len(doc.ents)
    length_score = len(claim.split()) / 10
    verb_score = any(token.pos_ == "VERB" for token in doc)

    return round(entity_score + length_score + (1 if verb_score else 0), 2)

In [35]:
def extract_claims_pipeline(text):

    # Step 1: LLM extraction
    claims = llm_extract_claims(text)

    # Step 2: Cleaning
    claims = clean_claims(claims)

    # Step 3: Remove hallucinations
    claims = remove_hallucinations(claims)

    # Step 4: Strict validation
    claims = strict_validation(claims, text)

    # Step 5: Scoring
    scored = [(c, score_claim(c)) for c in claims]

    return scored

In [36]:
results = extract_claims_pipeline(text)

for claim, score in results:
    print(f"{claim} --> Score: {score}")

The movie King will release on December 24, 2026. --> Score: 2.9
Shah Rukh Khan rejected a role in Jailer 2 to protect his aesthetic continuity for King. --> Score: 5.6
Suhana Khan is debuting alongside her father in a project directed by Siddharth Anand. --> Score: 4.4
Yaseen Khan claims the actor used eight passports for global travel due to international filming schedules. --> Score: 4.6
Shah Rukh Khan's franchise, Kolkata Knight Riders (KKR), has earned one point from five matches during the 2026 season. --> Score: 8.9
